## STOCHASTIC MODELING
MODULE 2 | LESSON 1


---


# **FOURIER-BASED OPTION PRICING**


|  |  |
|:---|:---|
|**Reading Time** | 100 minutes |
|**Prior Knowledge** | Derivative Pricing, Probability Density Function (PDF)  |
|**Keywords** | Fourier transform , Fast-Fourier Transform (FFT), Lewis (2001) |


---

In this module, we enter into probably the most important and problematic part of quantitative finance and derivative pricing: **model calibration**.

Before actually getting to calibrate models to real market data (will do this soon!), we need to take a small detour to fill our bagpack with a few important tools that are going to come in handy down the road. The first of these tools is **Fourier transforms**.

## **1. Why do we need Fourier methods?**

If you have a background in engineering or physics you may be familiar with Fourier methods. One common application of Fourier methods is to digital signal processing, enabling the analysis and manipulation of signals in the frequency domain. For instance, by allowing  the decomposition of complex signals into simpler sinusoidal components. This is not quite the use we are going to have for them, but Fourier methods are going to be our great ally for **model calibration**.



### 1.1. What is model calibration after all?

Calibrating a model is equivalent to finding those parameter values that best replicate **current market prices**.


- Why **current market prices**?

Prices for options in the market are determined by **supply & demand** from investors, which guarantees that these prices reflect investors' views (i.e., expectations) about the future.

Importantly, exchange-traded options are **vanilla ones** (simple European or American Calls & Puts), so we will always calibrate to these options. Once we have calibrated our model and obtained the proper parameter values, we can price any kind of exotic derivative instrument (e.g., Asian Call) using, for instance, Monte-Carlo techniques.


- Calibration ensures that we are in a **risk-neutral setting**

If you think about it, calibrating a model to market prices is equivalent to finding the Equivalent Martingale Measure (EMM) of the model (that is, our the set of **risk-neutral probabilities**).

Let's develop on this intuition in the simplest of settings. Think about a 1-step binomial model. The price today of a Call option is given by:

$$
C_0 = e^{-rT} ( p_u C_u + p_d C_d ) = e^{-rT} ( p_u C_u + (1-p_u) C_d )
$$

where,

$$
p_u = \frac{e^{rdt}- e^{-\sigma \sqrt{dt}}}{e^{\sigma \sqrt{dt}} - e^{-\sigma \sqrt{dt}}}
$$

\
Also, remember that we would know $r$, $T$, $dt$, and that we can get $C_u$ and $C_d$ via:

$$
C_u = u C_0 = e^{\sigma \sqrt{dt}} C_0 \ , \ and \ C_d = d C_0 = e^{-\sigma \sqrt{dt}} C_0
$$


\
Hence, the only **unknown** parameter for us is $\sigma$. If we get a good estimate for it, we would be able to perfectly replicate the risk-neutral probability density function (PDF), which in this case is simply based on $p_u$.

Model calibration, in this very simple setting, is achieved by obtaining the implied volatility in option market prices (i.e., $\sigma$). Remember this could be done with a basic Newton-Raphson algorithm, in which we iteratively gave values to $\sigma$ in the BS pricing equation until the model-produced prices are as close as possible to prices observed in the market.



- How does model calibration work in a **more complex framework**?

Intuitively, exactly the same. In a continuous-time framework the price of a Call option is given by:

$$
C_0 = e^{-r dt} \int_{0}^{∞} C_T(s) q(s) ds
$$

where $q(s)$ is the PDF of the underlying asset process and $C_T(s)$ corresponds to the possible payoffs at maturity.

The problem now is that $q(s)$ is not only composed by one unknown parameter but many, hence calibration is more difficult and important.

In a nutshell, **calibration is about** finding the set of parameter values for the model that minimize the distance between model-produced and market prices. In general form, we can calibrate a model with any optimization algorithm seeking to minimize, for instance, the **Mean Squared Error (MSE)**:

$$
\min MSE = \min \frac{1}{N} \sum ( C^* - \hat{C}(\alpha))^2
$$

\
where $C^*$ represents market prices (our target), and $\hat{C}(\alpha)$ the prices produced by the model under a set of parameter values in vector $\alpha$.

### 1.2. What's the use for Fourier methods here then?

From the previous example its easy to see that having a closed-form expression for the price of an option is going to simplify the calibration process dramatically. Here's when Fourier methods come into play.


In [1]:
from IPython.display import VimeoVideo

# Bigger video
VimeoVideo("1116720197", h="3298dbabb7", width=700, height=450)


For simple models, such Black-Scholes, we are able to obtain a closed-form expression for the price of an option. For example, using Ito's Lemma. But this is not entirely possible for more complex models (e.g., Heston), where **no closed-form solution** for the option price is attainable.

**Fourier methods** will help us circunvent this problem, as it has important advantages:

- *Generality*: it can be applied to the majority of existing (and yet to be created) models, with the only requirement of a **characteristic function** of the underlying process.

- *Accuracy*: while it does produce semi-analytical solutions that approximate the true one, these can be evaluated numerically, reaching a high degree of accuracy.

- *Speed*: this technique requires much lower computational costs than alternative like, for example, Monte Carlo methods.

Fourier methods were first introduced in option pricing in Heston (1993). But, before jumping there, we'll look at the use of this technique in a setting much more familiar to us: Black-Scholes.

## **2. Fourier methods**

In this section, we formally introduce Fourier transforms. The mathematical component here is relatively high, particularly for those of you that are not familiar with Fourier techniques. We will not show you here the detailed derivations or proofs, and rather focus on briefly describing the tool needed, but you can check them in the additional readings. We recommend you keep the focus on understanding the intuitive parts of the process, as these are the truly important ones.

### 2.1. Fourier Transforms

The **Fourier transform** of an integrable function $f(x)$ is:

\
$$
\hat{f}(u) = \int_{-\infty}^{\infty} e^{iux}f(x) \,dx
$$

where $u$ can be real or complex, and the $e^{iux}$ term is called *phase factor*.

\
**Fourier Inversion:** By Fourier inversion, we can recover the original function, $f(x)$:

\
$$
f(x) = \frac{1}{2\pi}\int_{-\infty}^{\infty} e^{-iux}\hat{f}(u) \,du
$$

\
(*Note that in some fields the Fourier expressions typically used have the reversed sign in the exponent as what we used here. That is, negative exponent in the Fourier transform ($e^{-iux}$) and positive in the inverse transform. This is not important and just a matter of notation standards. The important thing to keep in mind is that exponent changes signs from one to the other.*)

\
**Parseval's Relation:** Denotes inner product of two complex, square-integrable functions $f$ and $g$:

\
$$
\langle f,g \rangle = \int_{-\infty}^{\infty} f(x) \overline{g(x)} \,dx  \ = \frac{1}{2\pi} \int_{-\infty}^{\infty} \hat{f}(k) \overline{\hat{g}(k)} \,dk \ = \frac{1}{2\pi} \langle \hat{f}, \hat{g} \rangle
$$

\
All these concepts are going to be key in the application of Fourier methods to option pricing problems.

### 2.2. Fast Fourier Transform (FFT)

So far, the previous section covers Fourier transforms in a continuous setting. But there is also the alternative of doing things in discrete time, as we will show in the next sections. Here we briefly cover the Discrete Fourier Transform (DFT) and the Fast Fourier Transform (FFT), an efficient algorithm based on the former.

\
**Discrete Fourier Transform (DFT):** DFT can be generally defined as:

\
$$
\hat{x}(k) = \sum_{n=0}^{N} x(n) e^{\frac{-j2\pi kn}{N}}
$$

\
The upper bound $N$ will generally be a power of 2. Hence, the total number of operations to be computed will be $N^2$. Based on this idea, there is an efficient way to compute this sums.

\
The **Fast Fourier Transform (FFT)** is an efficient algorithm to compute these kinds of sums in DFT. Essentially, the algorithm computes two sums at the same time, reducing the number of operations from $N^2$ to $N ln_2 (N)$:

\
$$
\hat{x}[k] = \sum_{r=0}^{\frac{N}{2}-1} x[2r] e^{\frac{-j2\pi k(2r)}{N}} +  \sum_{r=0}^{\frac{N}{2}-1} x[2r+1] e^{\frac{-j2\pi k(2r+1)}{N}}
$$

\
We will explore FFT in detail later, as its the main ingredient of the **Carr-Madan method**, one of the semi-closed-form solutions proposed for vanilla options.

## **3. Pricing problem and Characteristic Function (CF)**

We know that the no-arbitrage value of a European Call is denoted by:

\
$$
C_0 = e^{-rT}\mathbf{E}^Q \left( S_T-K \right)^+  \  = \ e^{-rT} \int_0^\infty C_T(s) q(s) \ ds
$$

where $q(s)$ is the risk-neutral Probability Density Function (PDF) of $S_T$.
Unfortunately, the PDF of $S_T$ is not often known in closed-form in more complex models, unlike it happened in Black-Scholes. Fortunately, there is something that is known to us in closed-form, which is the **characteristic function (CF)** of $S_T$.

- What's a Characteristic Function (CF)?

The CF of any real-valued random variable *completely* defines its probability distribution.More important, conveniently for us, the CF of a random variable is simply the Fourier transform of its PDF.

Let random variable $X$ have PDF $q(x)$. Then, CF, $\hat{q}(u)$, will be:

\
$$
\hat{q}(u) = \int_{-\infty}^{\infty} e^{iux}q(x) \,dx = \mathbf{E}^Q \left( e^{iuX} \right)
$$

So, even if we do not know the PDF of random variable $S_t$, we do know its CF and, hence, can make our way back to its PDF for obtaining a semi-analytical solution for the price of the option.

This is precisely what the different option pricing methods based on Fourier transforms do. Let's look at them.

## **4. Fourier pricing**

There are several methods that rely on Fourier transforms to obtain analyitical solutions to option prices. Here we are going to take a look at the most famous and used ones. The first is Lewis (2001), based on continuous-time transforms. The second, Carr-Madan (1999) is based on discrete Fourier transforms.


### 4.1. Lewis (2001)

Lewis (2001) is probably the most common approach to obtain a (semi) closed-form expression for the price of an option under complex dynamics such stochastic volatility.

\
Consider a European Call option with payoff:

$$
C_T = max[e^s - K, 0] \ , \ where \ s = log S.
$$

\
For $u = u_r + iu_i$, with $u_i>1$, the **Fourier transform of $C_T$** is (you can check the additional references for proof):

$$
\hat{C}(u) = -\frac{K^{iu+1}}{u^2-iu}
$$

\
Using **Fourier inversion** on this expression yields:

$$
C_T(s) = \frac{1}{2\pi} \int_{-\infty+iu_i}^{\infty+iu_i} e^{-ius}\hat{C}_T(u) \,du
$$

\
Now, remember that the price of the option at $t=0$ is given by

\
$$
C_0 = e^{-rT}\mathbf{E}^Q(C_T) = \ \frac{e^{-rT}}{2\pi} \int_{-\infty+iu_i}^{\infty+iu_i}  \mathbf{E}^Q(e^{i(-u)s})\hat{C}_T(u) \,du \ =
\frac{e^{-rT}}{2\pi} \int_{-\infty+iu_i}^{\infty+iu_i}  \hat{C}_T(u) \hat{q}(-u) \,du
$$

\
\
If $S_t \equiv S_0 e^{-rt+X_t}$, with $X_t$ a Lèvy process and $e^{X_t}$ a martingale such that $X_0 = 0$, then:

\
$$
\hat{q}(-u) = e^{-iuy} \varphi(-u)
$$

where $\varphi$ is the **characteristic function** of $X_t$ and $y\equiv log S_0 + rT$.

\
Further defining $k = log(S_0/K)$ and assuming $u_i = 0.5 $, the present value of the option is:

\
$$
C_0 = S_0 - \frac{\sqrt{S_0 K} e^{-rT}}{\pi} \int_{0}^{\infty} \mathbf{Re}[e^{izk} \varphi(z-i/2)] \frac{dz}{z^2+1/4}
$$

\
where $\mathbf{Re[x]}$ denotes the real part of $x$.

(*Note that if you go to the original Lewis (2001) paper, this expression is a little bit different. It is mathematically equivalent though.*)



### 4.2. Carr-Madan (1999)

While in most cases, people rely on the Lewis (2001) model for a semi-analytical solution to the option pricing problem, there is another interesting approach by **Carr and Madan (1999)** that relies on FFT.

\
Carr and Madan (1999) define the classic European Call payoff

$$
C_T \equiv max[S_T-K,0]
$$

with $K\equiv e^k$ and $S_T \equiv e^s$. So, the expression becomes:

\
$$
C_0 \equiv e^{-rT}\mathbf{E}^Q \left( max[e^s - e^k, 0]\right) = e^{-rT} \int_{k}^{\infty} \left( e^s - e^k \right) q(s) \,ds
$$

\
with $q(s)$ risk-neutral prob of $s_T = log S_T$.

\
Defining $c_0 \equiv e^{\alpha k}C_0$, the Fourier transform becomes

\
$$
\Psi(\nu) \equiv  \int_{-\infty}^{\infty} e^{i\nu k} c_0 \,dk
$$

\
With inverse transform

\
$$
C_0 = \frac{e^{-\alpha k}}{\pi} \int_{0}^{\infty} e^{-i\nu k} \Psi(\nu) \,d\nu\
$$

\
and $\Psi(\nu)$
\
$$
\Psi(\nu) = \frac{e^{-rT} \varphi \left(\nu - (\alpha +1)i \right)}{\alpha^2 + \alpha - \nu^2 + i(2\alpha + 1)\nu}
$$

\
For the case of **In-The-Money (ITM)** options and characteristic function $\varphi (u) \equiv \mathbf{E}^Q(e^{ius_T})$, for $s_T \equiv log S_T$


## **5. Conclusion**

Arguably, this all has been mathematically dense. We will soon learn how to implement all this math in code, which will reduce the degree of abstraction. For now, please try to focus on understanding the use of Fourier methods here. You may want to come back here once you are dealing with the coding part for details of the implementation.



\
**References**

- Hilpisch, Yves. *Derivatives Analytics with Python: Data Analysis, Models, Simulation, Calibration and Hedging.* John Wiley & Sons, 2015.

- Lewis, Alan L. "A Simple Option Formula for General Jump-Diffusion and Other Exponential Lévy Processes." 2001.

- Carr, Peter, and Dilip Madan. "Option valuation using the fast Fourier transform." Journal of computational finance 2.4 (1999): 61-73.

---
Copyright 2025 WorldQuant University. This
content is licensed solely for personal use. Redistribution or
publication of this material is strictly prohibited.